In [1]:
import os
import numpy as np
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


In [36]:

# Define paths
train_dir = 'Training/'
test_dir = 'Testing/'
classes = ['glioma', 'meningioma', 'notumor', 'pituitary']


In [37]:

# Function to load images and labels
def load_images_and_labels(data_dir, classes, image_size=(64, 64)):
    X = []
    y = []
    for i, cls in enumerate(classes):
        cls_dir = os.path.join(data_dir, cls)
        for img_name in os.listdir(cls_dir):
            img_path = os.path.join(cls_dir, img_name)
            img = Image.open(img_path).convert('L')  # Convert to grayscale
            img = img.resize(image_size)  # Resize to fixed size
            img_array = np.array(img).flatten()  # Flatten the image
            X.append(img_array)
            y.append(i)
    return np.array(X), np.array(y)

In [38]:
# Load training set
print("Loading training set...")
X_train, y_train = load_images_and_labels(train_dir, classes, image_size=(64, 64))
print(f"Training set loaded: {X_train.shape[0]} samples, {X_train.shape[1]} features")


Loading training set...
Training set loaded: 5600 samples, 4096 features


In [39]:
# Apply Principal Component Analysis (PCA)
print("Applying PCA...")
n_components = 450  # You can adjust this number based on your needs
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train)
print(f"PCA applied: reduced to {n_components} components")

# Train Support Vector Machine (SVM)
print("Training SVM...")
svm = SVC(kernel='rbf', C=1.0, gamma='scale')  # You can tune these parameters
svm.fit(X_train_pca, y_train)
print("SVM trained successfully")


Applying PCA...
PCA applied: reduced to 450 components
Training SVM...
SVM trained successfully


In [40]:

# Load testing set for evaluation
print("Loading testing set...")
X_test, y_test = load_images_and_labels(test_dir, classes, image_size=(64, 64))
print(f"Testing set loaded: {X_test.shape[0]} samples")

# Apply PCA to test set
X_test_pca = pca.transform(X_test)

# Predict and evaluate
print("Evaluating on test set...")
y_pred = svm.predict(X_test_pca)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# Optional: Print classification report
from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=classes))

Loading testing set...
Testing set loaded: 1600 samples
Evaluating on test set...
Test Accuracy: 0.8506

Classification Report:
              precision    recall  f1-score   support

      glioma       0.83      0.69      0.76       400
  meningioma       0.83      0.78      0.80       400
     notumor       0.83      0.98      0.90       400
   pituitary       0.90      0.95      0.93       400

    accuracy                           0.85      1600
   macro avg       0.85      0.85      0.85      1600
weighted avg       0.85      0.85      0.85      1600



In [42]:
# Test set Macro ROC AUC
from sklearn.metrics import roc_curve, auc
y_test_bin = np.zeros((y_test.size, len(classes)))
for i in range(len(classes)):
    y_test_bin[:, i] = (y_test == i).astype(int)
y_score = svm.decision_function(X_test_pca)
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(len(classes)):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
print(f"Macro ROC AUC: {np.mean(list(roc_auc.values())):.4f}")



Macro ROC AUC: 0.9336
